# Mass-Resolution Force Correction — Validation Notebook

**Goal**: validate the mass-resolution force correction pipeline step by step.

| Section | Question |
|---------|----------|
| 1 | Is ΔF_mass well-defined and non-trivial? |
| 2 | Does ΔF_mass correlate with Lagrangian features? |
| 3 | Training monitoring (WandB) |
| 4 | Inference: does the MLP predict ΔF_mass? |
| 5 | Spatial validation — force maps |
| 6 | Generalisation across snapshots |
| 7 | Verdict |

**Key concepts**
- `F_LR`:   PM force on LR particle from 64³ density on 64³ mesh
- `F_HR(q)`: PM force on HR particle at same Lagrangian position q, from 128³ density on 128³ mesh
- `ΔF_mass = F_HR(q) − F_LR`  (mass-resolution force residual)

Physical effects captured beyond force-resolution correction:
- Reduced shot noise
- Small halos resolved only at HR mass resolution
- Accurate small-scale tidal fields


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import yaml
from types import SimpleNamespace
from pathlib import Path
import pickle
import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr

import sys
REPO_ROOT = Path("../").resolve()   # adjust if needed
sys.path.insert(0, str(REPO_ROOT / "pm2nbody"))

from train_lag_massres import (
    compute_massres_force_pair,
)
from train_lag_force import (
    load_snapshot,
    snapshot_features,
    compute_metrics,
)
from jaxpm.lagrangian import (
    get_axis_neighbor_indices,
    get_shell_neighbor_indices,
    make_lagrangian_corrector,
)
print("imports OK  |  JAX devices:", jax.devices())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIGURATION — edit this cell
# ═══════════════════════════════════════════════════════════════════
CONFIG_PATH = REPO_ROOT / "configs/lag_massres.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

data_cfg  = SimpleNamespace(**cfg["data"])
model_cfg = SimpleNamespace(**cfg["model"])

DATA_DIR   = Path(data_cfg.data_dir)
MESH_LR    = int(data_cfg.mesh_lr)
MESH_HR    = int(data_cfg.mesh_hr)
BOX_SIZE   = float(data_cfg.box_size)
N_PART     = int(data_cfg.n_particles)
R          = MESH_HR / MESH_LR          # resolution ratio

SIM_TRAIN  = int(data_cfg.sim_id_train)
SIM_VAL    = int(getattr(data_cfg, "sim_id_val", 1))
SNAP_TRAIN = int(data_cfg.snap_train)
SNAP_VAL   = int(data_cfg.snap_train)
SNAPS_VAL  = list(data_cfg.snaps_val)

USE_STRAIN     = bool(model_cfg.use_strain)
USE_INVARIANTS = bool(model_cfg.use_invariants)
USE_VELOCITY   = bool(model_cfg.use_velocity)
N_SHELL        = int(getattr(model_cfg, "n_shell", 0))
ENV_POOL_MODE  = str(getattr(model_cfg, "env_pool_mode", "mean_var"))

# Gaussian smoothing (in LR-cell units) — must match training config
# smooth_sigma_lr: applied to LR density  (0 = off, recommended to stay 0)
# smooth_sigma_hr: applied to HR density to reduce shot noise
#   typical: 0.5 → σ = R*0.5 HR cells  (e.g. 1 HR cell for R=2)
SMOOTH_SIGMA_LR = float(getattr(data_cfg, "smooth_sigma_lr", 0.0))
SMOOTH_SIGMA_HR = float(getattr(data_cfg, "smooth_sigma_hr", 0.0))

CHECKPOINT_DIR = REPO_ROOT / "runs/lag_massres"
SV_CENTER = [MESH_LR // 2] * 3
SV_HALF   = max(6, MESH_LR // 10)

print(f"mesh_lr={MESH_LR}  mesh_hr={MESH_HR}  r={R:.0f}  box={BOX_SIZE} Mpc/h")
print(f"train sim={SIM_TRAIN} snap={SNAP_TRAIN} | val sim={SIM_VAL} snaps={SNAPS_VAL}")
print(f"n_shell={N_SHELL}  env_pool_mode='{ENV_POOL_MODE}'")
print(f"smoothing: σ_lr={SMOOTH_SIGMA_LR} LR-cells  |  σ_hr={SMOOTH_SIGMA_HR} LR-cells "
      f"(= {SMOOTH_SIGMA_HR * R:.2f} HR-cells)")

In [ ]:
# ── Lagrangian neighbour indices (shared across all sections) ──────────────
neighbor_idx = get_axis_neighbor_indices(N_PART)

ext_neighbor_idx, ext_offsets, shell_slices = None, None, ()
if N_SHELL > 0:
    ext_neighbor_idx, ext_offsets, shell_slices = get_shell_neighbor_indices(
        N_PART, N_SHELL
    )
    K = (2 * N_SHELL + 1)**3 - 1
    print(f"Extended neighbourhood: n_shell={N_SHELL}  K={K}  pool='{ENV_POOL_MODE}'")
else:
    print("No extended neighbourhood (n_shell=0)")


def get_feats(pos_t):
    """Compute (feats, det_D) using the same settings as during training."""
    return snapshot_features(
        pos_t, neighbor_idx, MESH_LR, USE_STRAIN, USE_INVARIANTS,
        ext_neighbor_idx, ext_offsets, ENV_POOL_MODE, shell_slices,
    )


# ── Force pair helper ─────────────────────────────────────────────────────
# Uses SMOOTH_SIGMA_LR / SMOOTH_SIGMA_HR from the config cell — must match
# what was used during training for a fair evaluation of the target.
#
# Lagrangian correspondence verified:
#   LR particle m = ix·n_part² + iy·n_part + iz  starts at (ix, iy, iz) mesh_lr
#   HR particle at Lagrangian (ix·stride, iy·stride, iz·stride) mesh_hr starts at
#   the SAME physical position → stride-subsample [::stride,::stride,::stride] ✓
#   (stride = MESH_HR // N_PART)
_force_pair = jax.jit(lambda pl, ph: compute_massres_force_pair(
    pl, ph, N_PART, MESH_LR, MESH_HR, SMOOTH_SIGMA_LR, SMOOTH_SIGMA_HR
))

print(f"Force pair ready  (σ_lr={SMOOTH_SIGMA_LR}, σ_hr={SMOOTH_SIGMA_HR} LR-cells)")

## Section 1 — Force target: sanity & signal

In [ ]:
pos_lr_t, vel_lr_t, pos_hr_t, a_train = load_snapshot(
    DATA_DIR, SIM_TRAIN, SNAP_TRAIN, MESH_LR, MESH_HR, BOX_SIZE
)
assert pos_hr_t is not None, "HR positions not found — check DATA_DIR and MESH_HR"
print(f"a_train={a_train:.4f}  N_LR={pos_lr_t.shape[0]:,}  N_HR={pos_hr_t.shape[0]:,}")

feats_train, det_D_train = get_feats(pos_lr_t)
feat_dim_train = int(feats_train.shape[1])
strain_mag_train = np.sqrt(np.sum(np.asarray(feats_train[:, :9])**2, axis=-1))
sc_frac_train    = float(np.mean(det_D_train < 0))

f_lr_tr, f_hr_tr, delta_f_tr = _force_pair(pos_lr_t, pos_hr_t)

df_np     = np.asarray(jax.device_get(delta_f_tr))
f_hr_np   = np.asarray(jax.device_get(f_hr_tr))
f_lr_np   = np.asarray(jax.device_get(f_lr_tr))
df_mag    = np.sqrt(np.sum(df_np**2,   axis=-1))
f_hr_mag  = np.sqrt(np.sum(f_hr_np**2, axis=-1))
f_lr_mag  = np.sqrt(np.sum(f_lr_np**2, axis=-1))

print(f"\n|F_LR|   mean = {f_lr_mag.mean():.4e}")
print(f"|F_HR|   mean = {f_hr_mag.mean():.4e}")
print(f"|ΔF|     mean = {df_mag.mean():.4e}  ({df_mag.mean()/f_hr_mag.mean():.1%} of F_HR)")
print(f"SC frac = {sc_frac_train:.3%}")
print(f"Feature dim = {feat_dim_train}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ss = np.random.default_rng(0).choice(len(df_mag), min(50_000, len(df_mag)), replace=False)

# ΔF component distribution
for ci, (comp, color) in enumerate(zip(["x","y","z"],["C0","C1","C2"])):
    axes[0].hist(df_np[:, ci], bins=200, alpha=0.6, color=color,
                 density=True, label=f"dF_{comp}")
axes[0].set_xlabel("dF [mesh_lr units]")
axes[0].legend()
axes[0].set_title("Force correction distribution")

# |ΔF| vs |F_HR|
axes[1].hexbin(f_hr_mag[ss], df_mag[ss], gridsize=80,
               norm=LogNorm(), mincnt=1, cmap="plasma")
axes[1].set_xlabel("|F_HR| [mesh]"); axes[1].set_ylabel("|ΔF|")
axes[1].set_title(f"|ΔF| vs |F_HR|  a={a_train:.3f}")

# |ΔF| vs strain magnitude
r_strain, _ = pearsonr(strain_mag_train[ss], df_mag[ss])
axes[2].hexbin(strain_mag_train[ss], df_mag[ss], gridsize=80,
               norm=LogNorm(), mincnt=1, cmap="viridis")
axes[2].set_xlabel("||E||_F"); axes[2].set_ylabel("|ΔF|")
axes[2].set_title(f"R(strain, |ΔF|) = {r_strain:.4f}")

plt.suptitle(f"Mass-resolution force residual  a={a_train:.3f}", fontsize=12)
plt.tight_layout(); plt.show()
print("Signal verdict:", "PASS" if abs(r_strain) > 0.2 else "WEAK" if abs(r_strain) > 0.05 else "FAIL")

In [ ]:
# ── Section 1b: Effect of Gaussian smoothing on the HR force target ────────
# Shows how shot noise in the raw HR force is reduced by the Gaussian kernel.
# Only meaningful when SMOOTH_SIGMA_HR > 0.
from train_lag_massres import _gaussian_filter_k
from jaxpm.kernels import fftk
from jaxpm.pm import get_delta, potential_kgrid_to_force_at_pos

def compute_force_with_sigma(pos_lr, pos_hr, sigma_hr_lr_cells):
    """Compute HR force at LR Lagrangian positions with custom σ_hr (in LR-cell units)."""
    r      = MESH_HR / MESH_LR
    stride = MESH_HR // N_PART
    pos_hr_mesh = jnp.mod(pos_hr * r, MESH_HR)
    kvec_hr     = fftk((MESH_HR,) * 3)
    delta_hr_k  = jnp.fft.rfftn(get_delta(pos_hr_mesh, (MESH_HR,) * 3))
    if sigma_hr_lr_cells > 0:
        delta_hr_k = _gaussian_filter_k(delta_hr_k, kvec_hr, sigma_hr_lr_cells * r)
    f_hr_all = potential_kgrid_to_force_at_pos(delta_hr_k, pos_hr_mesh, kvec_hr) * r
    return f_hr_all.reshape(MESH_HR, MESH_HR, MESH_HR, 3)[::stride, ::stride, ::stride, :].reshape(-1, 3)

sigmas_to_compare = [0.0, 0.5, 1.0, 2.0]
f_hr_variants = {σ: np.asarray(jax.device_get(
    jax.jit(lambda ph: compute_force_with_sigma(pos_lr_t, ph, σ))(pos_hr_t)
)) for σ in sigmas_to_compare}

# Power spectra of |ΔF| = F_HR(σ) - F_LR for each sigma
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["C0", "C1", "C2", "C3"]

for (σ, f_hr_s), col in zip(f_hr_variants.items(), colors):
    df_s    = f_hr_s - f_lr_np
    df_s_mag = np.sqrt(np.sum(df_s**2, axis=-1))
    axes[0].hist(df_s[:, 0], bins=200, alpha=0.5, color=col, density=True,
                 label=f"σ_hr={σ} LR-cells ({σ*R:.1f} HR-cells)")
    # RMS of |ΔF| — scalar summary
    rms = df_s_mag.mean()
    axes[1].bar(f"σ={σ}", rms, color=col, alpha=0.8)
    axes[1].text(f"σ={σ}", rms * 1.01, f"{rms:.3f}", ha="center", va="bottom", fontsize=9)

axes[0].set_xlabel("ΔF_x component [mesh_lr units]")
axes[0].set_ylabel("density")
axes[0].set_title("Distribution of ΔF_x for different HR smoothings")
axes[0].legend(fontsize=8)

axes[1].set_ylabel("Mean |ΔF_mass| [mesh_lr units]")
axes[1].set_title("Signal strength vs smoothing σ_hr\n(smaller σ = more shot noise)")

plt.suptitle(
    f"Effect of Gaussian smoothing on HR force target  a={a_train:.3f}\n"
    f"(current config: σ_hr={SMOOTH_SIGMA_HR} LR-cells = {SMOOTH_SIGMA_HR*R:.1f} HR-cells)",
    fontsize=11
)
plt.tight_layout(); plt.show()

print(f"\nNote: training uses σ_hr={SMOOTH_SIGMA_HR} LR-cells  (σ={SMOOTH_SIGMA_HR*R:.1f} HR cells)")
print("  σ=0   → includes HR shot noise as part of target (noisy, harder to learn)")
print("  σ=0.5 → ~1 HR-cell smoothing (removes Nyquist noise, keeps signal)")
print("  σ≥1.0 → removes physical small-scale signal along with noise")

## Section 2 — Feature–force correlation

In [ ]:
feat_names = [f"E_{ab}" for ab in ["xx","xy","xz","yx","yy","yz","zx","zy","zz"]]
if USE_INVARIANTS:
    feat_names += ["tr(D)", "det(D)", "||E||_F"]
if N_SHELL > 0:
    n_env = feat_dim_train - len(feat_names)
    feat_names += [f"env_{i}" for i in range(n_env)]

feats_np = np.asarray(jax.device_get(feats_train))
n_feat   = feats_np.shape[1]
names_to_plot = feat_names[:n_feat]

r_vals = [pearsonr(feats_np[:, fi], df_mag)[0] for fi in range(n_feat)]

fig, ax = plt.subplots(figsize=(max(10, n_feat * 0.7), 4))
colors  = ["tomato" if abs(r) > 0.1 else "steelblue" for r in r_vals]
ax.bar(range(len(r_vals)), r_vals, color=colors)
ax.set_xticks(range(len(r_vals)))
ax.set_xticklabels(names_to_plot, rotation=45, ha="right")
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("Pearson R  with  |ΔF|")
ax.set_title(f"Feature–force correlation  (a={a_train:.3f})  red = |R|>0.1")
plt.tight_layout(); plt.show()

## Section 3 — Training monitoring

In [ ]:
history = None
try:
    import wandb
    api  = wandb.Api()
    runs = api.runs("pm2nbody_lag_massres", order="-created_at", per_page=1)
    run  = next(iter(runs))
    history = run.history(samples=2000)
    print(f"WandB run: {run.name}  ({len(history)} steps)")
except Exception as e:
    print(f"WandB not available: {e}")

if history is not None:
    import pandas as pd
    df_hist = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    if "train/loss"           in df_hist: axes[0].semilogy(df_hist["_step"], df_hist["train/loss"]);               axes[0].set_title("Train loss")
    if "train/pearson_r_mean" in df_hist: axes[1].plot(df_hist["_step"], df_hist["train/pearson_r_mean"]);         axes[1].set_title("Train R")
    if "val/force_mse_mean"   in df_hist: axes[2].semilogy(df_hist["_step"], df_hist["val/force_mse_mean"]);       axes[2].set_title("Val Force MSE")
    for ax in axes: ax.set_xlabel("step")
    plt.tight_layout(); plt.show()

## Section 4 — Inference: MLP force prediction

In [ ]:
PARAMS_LOADED = None

run_dirs = sorted(CHECKPOINT_DIR.glob("*/"), key=lambda p: p.stat().st_mtime, reverse=True)
for rd in run_dirs:
    for fname in ["best_params.pkl", "final_params.pkl"]:
        pkl = rd / fname
        if pkl.exists():
            with open(pkl, "rb") as fh:
                PARAMS_LOADED = hk.data_structures.to_immutable_dict(pickle.load(fh))
            print(f"Loaded: {pkl}"); break
    if PARAMS_LOADED is not None: break

if PARAMS_LOADED is None:
    print("No checkpoint found in", CHECKPOINT_DIR)

# Build model with same hidden_dim, n_layers, output_dim=3 as training
lag_model = make_lagrangian_corrector(
    hidden_dim=int(model_cfg.hidden_dim),
    n_layers=int(model_cfg.n_layers),
    output_dim=3,
)

# Validation snapshot
pos_v, vel_v, pos_hr_v, a_val = load_snapshot(
    DATA_DIR, SIM_VAL, SNAP_VAL, MESH_LR, MESH_HR, BOX_SIZE
)
feats_v, det_D_v = get_feats(pos_v)   # includes extended neighbourhood if n_shell>0
feat_dim_val     = int(feats_v.shape[1])
strain_mag_v     = np.sqrt(np.sum(np.asarray(feats_v[:, :9])**2, axis=-1))
sc_frac_v        = float(np.mean(det_D_v < 0))

_, _, delta_f_v = _force_pair(pos_v, pos_hr_v)
df_v_np  = np.asarray(jax.device_get(delta_f_v))
df_v_mag = np.sqrt(np.sum(df_v_np**2, axis=-1))
vel_feat_v = vel_v if USE_VELOCITY else jnp.zeros_like(vel_v)

print(f"Validation  a={a_val:.4f}  SC={sc_frac_v:.2%}  |ΔF| mean={df_v_mag.mean():.4e}")
print(f"Feature dim val = {feat_dim_val}  (train = {feat_dim_train})")
assert feat_dim_val == feat_dim_train, (
    f"Feature dim mismatch: val={feat_dim_val} != train={feat_dim_train}. "
    "Check n_shell / env_pool_mode in config."
)

In [ ]:
# ── CNN prior — two-stage accounting ──────────────────────────────────────
# If the MLP was trained as STAGE 2 (residual after CNN), the MLP output is:
#
#   pred_v_np  =  ΔF_MLP  =  ΔF_mass − ΔF_CNN          (residual target)
#
# So the TOTAL correction must be:
#   F_corrected  =  F_LR + ΔF_CNN + ΔF_MLP              (two-stage)
#   F_corrected  =  F_LR + ΔF_MLP                        (stage-1 only)
#
# We also adjust the scatter target in Section 4:
#   two-stage  → pred vs (ΔF_mass − ΔF_CNN)  [what the MLP was trained on]
#   stage-1    → pred vs  ΔF_mass             [unchanged]
#
# The function compute_cnn_massres_correction() from train_lag_massres reproduces
# exactly what was subtracted during training.

from train_lag_massres import (
    compute_cnn_massres_correction,
    _load_cnn_massres_checkpoint,
)

CNN_MODEL       = None
CNN_PARAMS      = None
IS_TWO_STAGE    = False

CNN_CKPT_PATH = getattr(model_cfg, "cnn_checkpoint", None)

if CNN_CKPT_PATH is not None:
    _ckpt_abs = str(REPO_ROOT / CNN_CKPT_PATH) if not Path(CNN_CKPT_PATH).is_absolute() else CNN_CKPT_PATH
    try:
        CNN_MODEL, CNN_PARAMS = _load_cnn_massres_checkpoint(_ckpt_abs)
        IS_TWO_STAGE = True
        print(f"CNN checkpoint loaded:  {_ckpt_abs}")
        n_cnn_params = sum(x.size for x in jax.tree_util.tree_leaves(CNN_PARAMS))
        print(f"  CNN params: {n_cnn_params:,}")
    except Exception as e:
        print(f"WARNING: Could not load CNN checkpoint: {e}")
        print("  Falling back to stage-1 only (f_corrected = F_LR + ΔF_MLP)")
else:
    print("No cnn_checkpoint in config → stage-1 MLP  (f_corrected = F_LR + ΔF_MLP)")

# ── Compute ΔF_CNN for the current validation snapshot ────────────────────
# Uses pos_v / vel_v / a_val defined in the cell above.
if IS_TWO_STAGE and CNN_MODEL is not None:
    _apply_cnn_jit = jax.jit(
        lambda pos, vel, a: compute_cnn_massres_correction(
            CNN_MODEL, CNN_PARAMS, pos, vel, a, MESH_LR
        )
    )
    cnn_delta_f_v_np = np.asarray(jax.device_get(
        _apply_cnn_jit(pos_v, vel_feat_v, jnp.array(a_val))
    ))
    print(f"\nΔF_CNN  mean |·| = {np.sqrt(np.sum(cnn_delta_f_v_np**2, axis=-1)).mean():.4e}")
    print(f"ΔF_mass mean |·| = {df_v_mag.mean():.4e}")
    cnn_frac = np.sqrt(np.sum(cnn_delta_f_v_np**2, axis=-1)).mean() / (df_v_mag.mean() + 1e-12)
    print(f"CNN covers {cnn_frac:.1%} of ΔF_mass")
else:
    cnn_delta_f_v_np = np.zeros_like(df_v_np)   # no CNN contribution

# ── Adjusted target for MLP scatter (Section 4) ───────────────────────────
# two-stage: MLP was trained on (ΔF_mass − ΔF_CNN)
# stage-1:   MLP was trained on  ΔF_mass
df_v_target_np = df_v_np - cnn_delta_f_v_np   # residual for two-stage, full for stage-1

stage_label = "F_LR + ΔF_CNN + ΔF_MLP" if IS_TWO_STAGE else "F_LR + ΔF_MLP"
print(f"\nTotal correction formula:  {stage_label}")
print(f"MLP scatter target:        {'ΔF_mass − ΔF_CNN  (residual)' if IS_TWO_STAGE else 'ΔF_mass  (full correction)'}")

In [ ]:
if PARAMS_LOADED is None:
    print("No checkpoint loaded.")
else:
    pred_v_np = np.asarray(jax.device_get(
        jax.jit(lag_model.apply)(PARAMS_LOADED, feats_v, vel_feat_v, jnp.array(a_val))
    ))

    # ── Evaluate MLP against its OWN training target ──────────────────────
    # two-stage:  target = ΔF_mass − ΔF_CNN  (df_v_target_np)
    # stage-1:    target = ΔF_mass            (df_v_np = df_v_target_np)
    # df_v_target_np is set in the cell above (CNN prior cell).
    err      = pred_v_np - df_v_target_np
    mse_tot  = float(np.mean(err**2))
    frac_mse = float(np.mean(np.sum(err**2, axis=1) / (np.sum(df_v_target_np**2, axis=1) + 1e-12)))
    r_xyz    = [pearsonr(pred_v_np[:, c], df_v_target_np[:, c])[0] for c in range(3)]
    r_mean   = float(np.mean(r_xyz))

    target_label = "ΔF_residual = ΔF_mass − ΔF_CNN" if IS_TWO_STAGE else "ΔF_mass"
    print(f"Target: {target_label}")
    print(f"MSE={mse_tot:.4e}  FracMSE={frac_mse:.4f}  "
          f"R(x,y,z)=({r_xyz[0]:.3f},{r_xyz[1]:.3f},{r_xyz[2]:.3f})  R_mean={r_mean:.4f}")

    COMPS = ["x", "y", "z"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    ss = np.random.default_rng(42).choice(len(pred_v_np), min(40_000, len(pred_v_np)), replace=False)
    for ci in range(3):
        tgt  = df_v_target_np[ss, ci]
        pred = pred_v_np[ss, ci]
        lim  = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15
        ax   = axes[ci]
        h    = ax.hexbin(tgt, pred, gridsize=70, cmap="Blues",
                         norm=LogNorm(), mincnt=1, extent=[-lim, lim, -lim, lim])
        plt.colorbar(h, ax=ax)
        ax.plot([-lim, lim], [-lim, lim], "r--", lw=1)
        ax.set_xlabel(f"Target {COMPS[ci]}  ({target_label})")
        ax.set_ylabel(f"Pred  ΔF_{COMPS[ci]}")
        ax.set_title(f"{COMPS[ci]}  R={pearsonr(tgt, pred)[0]:.4f}")
    plt.suptitle(
        f"MLP pred vs target  a={a_val:.3f}"
        + ("  [two-stage: MLP predicts CNN residual]" if IS_TWO_STAGE else "  [stage-1: MLP predicts full ΔF]"),
        fontsize=11
    )
    plt.tight_layout(); plt.show()

    # ── For two-stage: also show total correction vs ΔF_mass ─────────────
    if IS_TWO_STAGE:
        total_pred_np = cnn_delta_f_v_np + pred_v_np   # ΔF_CNN + ΔF_MLP
        r_total = [pearsonr(total_pred_np[:, c], df_v_np[:, c])[0] for c in range(3)]
        r_total_mean = float(np.mean(r_total))
        print(f"\nTotal correction  (ΔF_CNN + ΔF_MLP) vs ΔF_mass:")
        print(f"  R(x,y,z) = ({r_total[0]:.3f}, {r_total[1]:.3f}, {r_total[2]:.3f})  R_mean={r_total_mean:.4f}")

        fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
        for ci in range(3):
            tgt_f  = df_v_np[ss, ci]
            pred_f = total_pred_np[ss, ci]
            lim_f  = max(abs(np.percentile(tgt_f, 1)), abs(np.percentile(tgt_f, 99))) * 1.15
            ax2    = axes2[ci]
            h2     = ax2.hexbin(tgt_f, pred_f, gridsize=70, cmap="Greens",
                                norm=LogNorm(), mincnt=1, extent=[-lim_f, lim_f, -lim_f, lim_f])
            plt.colorbar(h2, ax=ax2)
            ax2.plot([-lim_f, lim_f], [-lim_f, lim_f], "r--", lw=1)
            ax2.set_xlabel(f"ΔF_mass_{COMPS[ci]}")
            ax2.set_ylabel(f"ΔF_CNN + ΔF_MLP  {COMPS[ci]}")
            ax2.set_title(f"{COMPS[ci]}  R={pearsonr(tgt_f, pred_f)[0]:.4f}")
        plt.suptitle(
            f"Total correction (CNN + MLP) vs ΔF_mass  a={a_val:.3f}", fontsize=11
        )
        plt.tight_layout(); plt.show()

## Section 5 — Spatial validation: force maps

In [ ]:
if PARAMS_LOADED is not None:
    # Eulerian sub-volume based on LR positions (same idx_sv for all panels)
    pos_v_np = np.asarray(jax.device_get(pos_v)) % MESH_LR
    sv_mask  = np.all(np.abs(pos_v_np - np.array(SV_CENTER)) < SV_HALF, axis=1)
    idx_sv   = np.where(sv_mask)[0]
    px, py   = pos_v_np[idx_sv, 0], pos_v_np[idx_sv, 1]
    print(f"Sub-volume: {idx_sv.sum()} particles  (center={SV_CENTER}, half={SV_HALF})")

    pred_mag = np.sqrt(np.sum(pred_v_np**2, axis=-1))
    err_mag  = np.sqrt(np.sum((pred_v_np - df_v_np)**2, axis=-1))

    vmax = np.percentile(df_v_mag, 95)
    panels = [
        (df_v_mag, "|ΔF_target|",  "Oranges"),
        (pred_mag, "|ΔF_pred|",    "Blues"),
        (err_mag,  "|error|",       "hot"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ci, (mag, label, cmap) in enumerate(panels):
        ax = axes[ci]
        hb = ax.hexbin(px, py, C=mag[idx_sv], gridsize=60, cmap=cmap,
                       reduce_C_function=np.mean, vmin=0,
                       vmax=vmax if ci < 2 else np.percentile(mag, 95))
        plt.colorbar(hb, ax=ax, label=label)
        ax.set_title(label); ax.set_xlabel("x")
    plt.suptitle(f"Force maps  a={a_val:.3f}  SC={sc_frac_v:.2%}", fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
if PARAMS_LOADED is not None:
    # ── Recompute force pair on validation snapshot (fresh) ───────────────
    f_lr_v, f_hr_v, _ = _force_pair(pos_v, pos_hr_v)
    f_lr_v_np  = np.asarray(jax.device_get(f_lr_v))
    f_hr_v_np  = np.asarray(jax.device_get(f_hr_v))

    # ── Total corrected force ─────────────────────────────────────────────
    # two-stage:  F_LR + ΔF_CNN + ΔF_MLP
    # stage-1:    F_LR + ΔF_MLP   (cnn_delta_f_v_np = 0)
    f_corrected = f_lr_v_np + cnn_delta_f_v_np + pred_v_np

    f_lr_mag_v  = np.sqrt(np.sum(f_lr_v_np**2,   axis=-1))
    f_hr_mag_v  = np.sqrt(np.sum(f_hr_v_np**2,   axis=-1))
    f_cor_mag_v = np.sqrt(np.sum(f_corrected**2, axis=-1))

    vmax_f = np.percentile(f_hr_mag_v, 97)
    labels_f = ["F_LR", stage_label, "F_HR (reference)"]
    mags_f   = [f_lr_mag_v, f_cor_mag_v, f_hr_mag_v]
    cmaps_f  = ["Blues", "Greens", "Reds"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ci, (mag, label, cmap) in enumerate(zip(mags_f, labels_f, cmaps_f)):
        ax = axes[ci]
        hb = ax.hexbin(px, py, C=mag[idx_sv], gridsize=60, cmap=cmap,
                       reduce_C_function=np.mean, vmin=0, vmax=vmax_f)
        plt.colorbar(hb, ax=ax, label="|F|")
        ax.set_title(label); ax.set_xlabel("x")

    # Improvement metric: MSE of corrected force vs HR
    mse_lr  = float(np.mean((f_lr_v_np  - f_hr_v_np)**2))
    mse_cor = float(np.mean((f_corrected - f_hr_v_np)**2))
    improv  = 1.0 - mse_cor / mse_lr
    plt.suptitle(
        f"{stage_label}\n"
        f"|MSE_LR|={mse_lr:.4e}  |MSE_corrected|={mse_cor:.4e}  improvement={improv:.1%}",
        fontsize=10
    )
    plt.tight_layout(); plt.show()
    print(f"Force MSE improvement ({stage_label}): {improv:.1%}")

## Section 5b — Global force comparison & improvement metrics

Three diagnostic panels over **all** particles (not just a sub-volume):

| Panel | Question |
|-------|----------|
| 5b-1 | Does the correction shift |F| distribution toward HR? |
| 5b-2 | Do individual particles move closer to F_HR? (scatter below diagonal = improvement) |
| 5b-3 | CDF of |F − F_HR| — what fraction of particles benefit? |
| 5b-4 | Summary table — MSE / MAE / percentile errors |

In [ ]:
# ── 5b-1  Global force magnitude distributions ────────────────────────────
# Overlaid histograms of |F_LR|, |F_corrected|, |F_HR| over all N particles.
# If the correction works, the corrected histogram should shift toward F_HR.
if PARAMS_LOADED is not None:
    bins = np.linspace(0, np.percentile(f_hr_mag_v, 99.5), 120)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- linear scale ---
    ax = axes[0]
    ax.hist(f_lr_mag_v,  bins=bins, alpha=0.5, color="steelblue",  density=True, label="|F_LR|")
    ax.hist(f_cor_mag_v, bins=bins, alpha=0.5, color="seagreen",   density=True, label="|F_LR + ΔF_pred|")
    ax.hist(f_hr_mag_v,  bins=bins, alpha=0.5, color="tomato",     density=True, label="|F_HR|")
    ax.set_xlabel("|F| [mesh_lr units]")
    ax.set_ylabel("density")
    ax.set_title("Force magnitude distributions (linear)")
    ax.legend()

    # --- log scale: better view of the tails ---
    ax = axes[1]
    log_bins = np.logspace(
        np.log10(max(np.percentile(f_hr_mag_v, 1), 1e-6)),
        np.log10(np.percentile(f_hr_mag_v, 99.9)),
        120
    )
    ax.hist(f_lr_mag_v,  bins=log_bins, alpha=0.5, color="steelblue",  density=True, label="|F_LR|")
    ax.hist(f_cor_mag_v, bins=log_bins, alpha=0.5, color="seagreen",   density=True, label="|F_LR + ΔF_pred|")
    ax.hist(f_hr_mag_v,  bins=log_bins, alpha=0.5, color="tomato",     density=True, label="|F_HR|")
    ax.set_xscale("log"); ax.set_xlabel("|F| [mesh_lr units] (log)")
    ax.set_title("Force magnitude distributions (log scale)")
    ax.legend()

    # Means as vertical lines
    for ax_ in axes:
        ax_.axvline(f_lr_mag_v.mean(),  color="steelblue", ls="--", lw=1.2, alpha=0.9)
        ax_.axvline(f_cor_mag_v.mean(), color="seagreen",  ls="--", lw=1.2, alpha=0.9)
        ax_.axvline(f_hr_mag_v.mean(),  color="tomato",    ls="--", lw=1.2, alpha=0.9)

    plt.suptitle(
        f"Global |F| distribution — {len(f_lr_mag_v):,} particles  |  a={a_val:.3f}",
        fontsize=12
    )
    plt.tight_layout(); plt.show()

    print(f"Mean |F_LR|      = {f_lr_mag_v.mean():.4e}")
    print(f"Mean |F_corrected| = {f_cor_mag_v.mean():.4e}")
    print(f"Mean |F_HR|      = {f_hr_mag_v.mean():.4e}")

In [ ]:
# ── 5b-2  Per-particle improvement scatter ────────────────────────────────
# x-axis: |F_LR − F_HR|  (error before correction)
# y-axis: |F_corrected − F_HR|  (error after correction)
# Points BELOW the diagonal → correction helped.
# Points ABOVE the diagonal → correction hurt (over/under-shoot).
# Colour = log particle count per hexbin.
if PARAMS_LOADED is not None:
    err_lr_v  = np.sqrt(np.sum((f_lr_v_np  - f_hr_v_np)**2, axis=-1))
    err_cor_v = np.sqrt(np.sum((f_corrected - f_hr_v_np)**2, axis=-1))

    frac_improved = float(np.mean(err_cor_v < err_lr_v))
    mean_gain_lr  = float(np.mean(err_lr_v))
    mean_gain_cor = float(np.mean(err_cor_v))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ── scatter (hexbin) ──────────────────────────────────────────────────
    ax = axes[0]
    lim = np.percentile(err_lr_v, 99)
    h = ax.hexbin(err_lr_v, err_cor_v, gridsize=80, cmap="plasma",
                  norm=LogNorm(), mincnt=1,
                  extent=[0, lim, 0, lim])
    plt.colorbar(h, ax=ax, label="particle count")
    ax.plot([0, lim], [0, lim], "w--", lw=1.5, label="diagonal (no change)")
    ax.set_xlabel("|F_LR − F_HR|  (error before)")
    ax.set_ylabel("|F_corrected − F_HR|  (error after)")
    ax.set_title(
        f"Per-particle improvement\n"
        f"{frac_improved:.1%} of particles improved  |  "
        f"mean error: {mean_gain_lr:.3e} → {mean_gain_cor:.3e}"
    )
    ax.legend(fontsize=8)

    # ── relative improvement histogram ────────────────────────────────────
    # rel_gain > 0 means the correction reduced the error
    ax = axes[1]
    rel_gain = (err_lr_v - err_cor_v) / (err_lr_v + 1e-12)   # in [−∞, 1]
    clip = 1.5
    rel_gain_clipped = np.clip(rel_gain, -clip, clip)
    ax.hist(rel_gain_clipped, bins=150, color="seagreen", alpha=0.7, density=True)
    ax.axvline(0, color="k",      lw=1.5, ls="--", label="no change")
    ax.axvline(rel_gain.mean(), color="tomato", lw=1.5, label=f"mean={rel_gain.mean():.3f}")
    ax.set_xlabel("(|err_LR| − |err_corrected|) / |err_LR|  (positive = improvement)")
    ax.set_ylabel("density")
    ax.set_title("Per-particle relative error change")
    ax.legend()

    plt.suptitle(
        f"Per-particle force error comparison  |  a={a_val:.3f}  |  "
        f"{len(err_lr_v):,} particles",
        fontsize=12
    )
    plt.tight_layout(); plt.show()
    print(f"Fraction of particles with reduced error: {frac_improved:.2%}")

In [ ]:
# ── 5b-3  Force error CDF ─────────────────────────────────────────────────
# Cumulative distribution of |F − F_HR| for LR and corrected.
# A curve shifted LEFT = better (errors are smaller for more particles).
# Also shows breakdown: shell-crossing vs. non-SC particles.
if PARAMS_LOADED is not None:
    det_D_v_np = np.asarray(jax.device_get(det_D_v))
    sc_mask    = det_D_v_np < 0

    # Threshold range: 0 → 99th percentile of LR errors
    thresholds = np.linspace(0, np.percentile(err_lr_v, 99), 500)

    def cdf(errors, thresholds):
        return np.array([np.mean(errors <= t) for t in thresholds])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── all particles ─────────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(thresholds, cdf(err_lr_v,  thresholds), color="steelblue", lw=2, label="|F_LR − F_HR|")
    ax.plot(thresholds, cdf(err_cor_v, thresholds), color="seagreen",  lw=2, label="|F_corrected − F_HR|")
    ax.set_xlabel("|F − F_HR| [mesh_lr units]")
    ax.set_ylabel("CDF  P(error ≤ threshold)")
    ax.set_title(f"Force error CDF — all {len(err_lr_v):,} particles  (a={a_val:.3f})")
    ax.legend(); ax.grid(alpha=0.3)

    # Add 50th / 90th / 95th percentile markers
    for pct in [50, 90, 95]:
        p_lr  = np.percentile(err_lr_v,  pct)
        p_cor = np.percentile(err_cor_v, pct)
        ax.axvline(p_lr,  color="steelblue", ls=":", alpha=0.6)
        ax.axvline(p_cor, color="seagreen",  ls=":", alpha=0.6)

    # ── shell-crossing vs. non-SC ──────────────────────────────────────────
    ax = axes[1]
    for mask, label, ls in [
        (~sc_mask, "non-SC",  "-"),
        ( sc_mask, "SC only", "--"),
    ]:
        if mask.sum() > 0:
            ax.plot(thresholds, cdf(err_lr_v[mask],  thresholds),
                    color="steelblue", ls=ls, lw=2, label=f"|F_LR|  {label}")
            ax.plot(thresholds, cdf(err_cor_v[mask], thresholds),
                    color="seagreen",  ls=ls, lw=2, label=f"|F_cor| {label}")

    ax.set_xlabel("|F − F_HR| [mesh_lr units]")
    ax.set_ylabel("CDF")
    ax.set_title(f"CDF split: SC ({sc_mask.mean():.1%}) vs non-SC")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.suptitle("Force error CDFs  (left-shifted = better)", fontsize=12)
    plt.tight_layout(); plt.show()

    # Print key percentiles
    print(f"\n{'Metric':<30} {'LR':>12} {'Corrected':>12} {'Change':>10}")
    print("-" * 66)
    for pct in [50, 75, 90, 95]:
        p_lr  = np.percentile(err_lr_v,  pct)
        p_cor = np.percentile(err_cor_v, pct)
        chg   = (p_cor - p_lr) / p_lr * 100
        print(f"  P{pct:02d} |F error|          {p_lr:>12.4e} {p_cor:>12.4e} {chg:>9.1f}%")

In [ ]:
# ── 5b-4  Global improvement summary table ────────────────────────────────
# Comprehensive per-snapshot statistics: MSE, MAE, percentile errors, R.
# For two-stage training: total correction = F_LR + ΔF_CNN + ΔF_MLP.
# For stage-1:            total correction = F_LR + ΔF_MLP.
if PARAMS_LOADED is not None:
    import pandas as pd

    def snapshot_improvement(snap_id, sim_id=SIM_VAL):
        vpos, vvel, vpos_hr_s, va = load_snapshot(
            DATA_DIR, sim_id, snap_id, MESH_LR, MESH_HR, BOX_SIZE
        )
        vfeats, vdet_D = get_feats(vpos)
        vf_lr, vf_hr, vdf = _force_pair(vpos, vpos_hr_s)
        vvel_feat = vvel if USE_VELOCITY else jnp.zeros_like(vvel)
        vpred = jax.jit(lag_model.apply)(PARAMS_LOADED, vfeats, vvel_feat, jnp.array(va))

        vf_lr_np  = np.asarray(jax.device_get(vf_lr))
        vf_hr_np  = np.asarray(jax.device_get(vf_hr))
        vpred_np  = np.asarray(jax.device_get(vpred))
        vdf_np    = np.asarray(jax.device_get(vdf))    # full ΔF_mass

        # ── CNN contribution for this snapshot ────────────────────────────
        if IS_TWO_STAGE and CNN_MODEL is not None:
            vcnn_np = np.asarray(jax.device_get(
                jax.jit(lambda p, v, a: compute_cnn_massres_correction(
                    CNN_MODEL, CNN_PARAMS, p, v, a, MESH_LR
                ))(vpos, vvel_feat, jnp.array(va))
            ))
        else:
            vcnn_np = np.zeros_like(vpred_np)

        # ── Total corrected force: F_LR + ΔF_CNN + ΔF_MLP ────────────────
        vf_cor_np  = vf_lr_np + vcnn_np + vpred_np

        err_lr  = np.sqrt(np.sum((vf_lr_np  - vf_hr_np)**2, axis=-1))
        err_cor = np.sqrt(np.sum((vf_cor_np - vf_hr_np)**2, axis=-1))

        mse_lr   = float(np.mean((vf_lr_np  - vf_hr_np)**2))
        mse_cor  = float(np.mean((vf_cor_np - vf_hr_np)**2))
        mae_lr   = float(np.mean(err_lr))
        mae_cor  = float(np.mean(err_cor))
        sc_frac  = float(np.mean(np.asarray(jax.device_get(vdet_D)) < 0))

        # R of total correction vs ΔF_mass
        total_pred = vcnn_np + vpred_np
        r_total = float(np.mean([
            pearsonr(total_pred[:, c], vdf_np[:, c])[0] for c in range(3)
        ]))
        # R of MLP alone vs its own target (residual for stage-2)
        vdf_target = vdf_np - vcnn_np
        r_mlp = float(np.mean([
            pearsonr(vpred_np[:, c], vdf_target[:, c])[0] for c in range(3)
        ]))

        return {
            "snap":              snap_id,
            "a":                 float(va),
            "SC %":              sc_frac * 100,
            "MSE_LR":            mse_lr,
            "MSE_cor":           mse_cor,
            "MSE_improv %":      (1 - mse_cor / mse_lr) * 100,
            "MAE_LR":            mae_lr,
            "MAE_cor":           mae_cor,
            "MAE_improv %":      (1 - mae_cor / mae_lr) * 100,
            "P50_LR":            np.percentile(err_lr,  50),
            "P50_cor":           np.percentile(err_cor, 50),
            "P95_LR":            np.percentile(err_lr,  95),
            "P95_cor":           np.percentile(err_cor, 95),
            "frac_improved":     float(np.mean(err_cor < err_lr)) * 100,
            "R_total_vs_dF":     r_total,     # (ΔF_CNN + ΔF_MLP) vs ΔF_mass
            "R_mlp_vs_residual": r_mlp,       # ΔF_MLP vs its own training target
        }

    rows_5b = [snapshot_improvement(s) for s in SNAPS_VAL]
    df_5b   = pd.DataFrame(rows_5b)

    # ── print full table ───────────────────────────────────────────────────
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    print(f"Global force improvement statistics  [{stage_label}]")
    print("=" * 95)
    print(df_5b.to_string(index=False))
    print()

    # ── visualise improvement vs scale factor ─────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(df_5b["a"], df_5b["MSE_improv %"], "o-", color="seagreen", lw=2)
    axes[0].axhline(0, color="k", ls="--", lw=0.8)
    axes[0].set_xlabel("a"); axes[0].set_ylabel("MSE improvement %")
    axes[0].set_title("MSE improvement vs scale factor")
    axes[0].grid(alpha=0.3)

    axes[1].plot(df_5b["a"], df_5b["MAE_improv %"], "o-", color="steelblue", lw=2)
    axes[1].axhline(0, color="k", ls="--", lw=0.8)
    axes[1].set_xlabel("a"); axes[1].set_ylabel("MAE improvement %")
    axes[1].set_title("MAE improvement vs scale factor")
    axes[1].grid(alpha=0.3)

    axes[2].plot(df_5b["a"], df_5b["frac_improved"], "o-", color="tomato", lw=2)
    axes[2].axhline(50, color="k", ls="--", lw=0.8, label="50% baseline")
    axes[2].set_xlabel("a"); axes[2].set_ylabel("% particles improved")
    axes[2].set_title("% particles with reduced |F − F_HR|")
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

    plt.suptitle(
        f"Mass-resolution force correction — global metrics  [{stage_label}]",
        fontsize=11
    )
    plt.tight_layout(); plt.show()

    # ── mini-summary ───────────────────────────────────────────────────────
    print(f"\nSummary (mean over {len(SNAPS_VAL)} val snapshots)  [{stage_label}]:")
    print(f"  MSE improvement      : {df_5b['MSE_improv %'].mean():.1f}%")
    print(f"  MAE improvement      : {df_5b['MAE_improv %'].mean():.1f}%")
    print(f"  % particles improved : {df_5b['frac_improved'].mean():.1f}%")
    print(f"  R (total vs ΔF_mass) : {df_5b['R_total_vs_dF'].mean():.4f}")
    if IS_TWO_STAGE:
        print(f"  R (MLP vs residual)  : {df_5b['R_mlp_vs_residual'].mean():.4f}")
    print()
    print("Interpretation:")
    print("  MSE_improv > 0      → total correction reduces force error vs F_HR")
    print("  frac_improved > 50% → majority of particles benefit")
    if IS_TWO_STAGE:
        print("  R_mlp_vs_residual   → how well MLP captures what CNN missed")
        print("  R_total_vs_dF       → combined pipeline quality")

## Section 6 — Generalisation across snapshots

In [ ]:
if PARAMS_LOADED is not None:
    import pandas as pd
    rows = []
    for vsnap in SNAPS_VAL:
        vpos, vvel, vpos_hr_s, va = load_snapshot(
            DATA_DIR, SIM_VAL, vsnap, MESH_LR, MESH_HR, BOX_SIZE
        )
        vfeats, vdet_D = get_feats(vpos)   # includes extended neighbourhood if n_shell>0
        vf_lr, vf_hr, vdf = _force_pair(vpos, vpos_hr_s)

        vvel_feat = vvel if USE_VELOCITY else jnp.zeros_like(vvel)
        vpred = jax.jit(lag_model.apply)(PARAMS_LOADED, vfeats, vvel_feat, jnp.array(va))

        vf_lr_np  = np.asarray(jax.device_get(vf_lr))
        vf_hr_np  = np.asarray(jax.device_get(vf_hr))
        vpred_np  = np.asarray(jax.device_get(vpred))
        vdf_np    = np.asarray(jax.device_get(vdf))

        met = compute_metrics(vpred, vdf, vdet_D, prefix="")

        # Force MSE improvement
        mse_lr_v  = float(np.mean((vf_lr_np - vf_hr_np)**2))
        vf_cor    = vf_lr_np + vpred_np
        mse_cor_v = float(np.mean((vf_cor   - vf_hr_np)**2))
        improv_v  = 1.0 - mse_cor_v / mse_lr_v

        rows.append({
            "snap":          vsnap,
            "a":             float(va),
            "R_mean":        met["pearson_r_mean"],
            "frac_mse":      met["frac_mse"],
            "mse_lr":        mse_lr_v,
            "mse_corrected": mse_cor_v,
            "improvement_%": improv_v * 100,
        })
        del vpos, vvel, vfeats, vdf, vpred

    df_res = pd.DataFrame(rows)
    print(df_res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(df_res["a"], df_res["R_mean"],        "o-"); axes[0].set_title("R_mean vs a")
    axes[1].semilogy(df_res["a"], df_res["frac_mse"],  "o-"); axes[1].set_title("FracMSE vs a")
    axes[2].plot(df_res["a"], df_res["improvement_%"], "o-"); axes[2].set_title("Force MSE improvement % vs a")
    axes[2].axhline(0, color="k", lw=0.5)
    for ax in axes: ax.set_xlabel("scale factor a")
    plt.tight_layout(); plt.show()

## Section 7 — Verdict

In [ ]:
if PARAMS_LOADED is not None:
    print("="*60)
    print("  MASS-RESOLUTION FORCE CORRECTION — VERDICT")
    print("="*60)
    print(f"  R_mean (val)        : {r_mean:.4f}")
    print(f"  FracMSE (val)       : {frac_mse:.4f}")
    print(f"  Force MSE improv    : {improv:.1%}")
    if "df_res" in dir():
        print(f"  R across snaps      : {list(df_res.R_mean.round(3).values)}")
        print(f"  Improvement % snaps : {list(df_res['improvement_%'].round(1).values)}")
    print()
    checks = [
        (r_mean > 0.4,   f"R_mean={r_mean:.3f} > 0.4  (force residual predictive)"),
        (frac_mse < 0.8, f"FracMSE={frac_mse:.3f} < 0.8"),
        (improv > 0,     f"force MSE improves by {improv:.1%}"),
    ]
    for ok, msg in checks:
        print(f"  {'OK' if ok else 'XX'} {msg}")
    print("="*60)
    print()
    print("Notes:")
    print("  - R_mean > 0.4 is harder to achieve for mass-res than force-res")
    print("  - (mass-res targets small-scale non-linear effects, harder to predict)")
    print("  - MSE improvement is the key metric: even R~0.3 can give >20% MSE gain")